# **CSC 425 Course Project**
**Tic-Tac-Toe Game**

**Minimax Search Algorithm**

**Sydney Boomer & Cade Lederer**

***Minimax Algorithm***

The *Minimax* algorithm is a type of turn-based adversarial search in which two agents have conflicting goals. The two agents are characterized as a maximizer who is interested in deciding upon the maximum score available, and a minimizer who inversely decides upon the minimum score available. (In this implementation involving the AI vs. a human agent, the part of the algorithm that chooses the maximum score available is tied to the AI's decision making, making the user act as the minimizer). The algorithm explores a game tree, designed to incorporate the AI's and user's arrangement of moves as possible states, ending when a terminal state is reached.

Upon each search, each terminal state is assigned a utility value which represents the win/draw/loss status from the AI's perspective. A winning utility value is calculated using *10 -- depth*, where depth is the depth level of a specific terminal state in the game tree. This makes the algorithm prefer winning sooner (a quicker win will recieve a higher utility, as this state will take fewer moves to reach). These utility values are passed up through the game tree so that the AI is able to choose a child with the highest value it can guarantee (a move that increases the score) for any of its turns, or the user to choose a child with a move that decreases the score, whether that is the best move for the user or not.

***Game Definition***
*   State: arrangement of user and AI's moves (X's and O's) on a 3x3 tic-tac-toe game board
*   Start state: an empty 3x3 game board
*   Actions: AI and user input take turns to place X/O marks on an unoccupied tile
*   Constraints: moves performed by user and AI must be legal (e.g., X's and O's within 3x3 board, no placing an X or O on an occupied tile, user and AI take turns)
*   Terminal: the game reaches a terminal state (game has ended) when either the user or AI has won the game, defined by the 8 possible winning configurations for each player, or no more legal moves are available to either the user or AI (ie. a draw)
*   Utility: the utility function is defined by the final score when a terminal state is reached -- represents a win, draw, or loss for the AI, and how soon this result is reached.






In [ ]:
# define state space representation

EMPTY = " "
X_MARK = "X"
O_MARK = "O"
PLAYER_MARKS = (X_MARK, O_MARK)
BOARD_SIZE = 9

# each triple lists board conditions that form a winning row, column, or diagonal.
WIN_CONDITIONS = (
    (0, 1, 2), (3, 4, 5), (6, 7, 8),  # rows
    (0, 3, 6), (1, 4, 7), (2, 5, 8),  # columns
    (0, 4, 8), (2, 4, 6)              # diagonals
)

In [ ]:
# helper functions

def print_board(board):
    # display the board, showing position numbers for empty squares.
    display = [str(i + 1) if cell == EMPTY else cell for i, cell in enumerate(board)]
    print(f"\n {display[0]} | {display[1]} | {display[2]} ")
    print("---|---|---")
    print(f" {display[3]} | {display[4]} | {display[5]} ")
    print("---|---|---")
    print(f" {display[6]} | {display[7]} | {display[8]} \n")

def check_winner(board, player):
    # return True when player has three matching marks in a winning line.
    return any(all(board[index] == player for index in line) for line in WIN_CONDITIONS)

def is_draw(board):
    # return True when the board is full and neither mark has won.
    return (EMPTY not in board
            and not check_winner(board, X_MARK)
            and not check_winner(board, O_MARK))

In [ ]:
# state class

class TicTacToeState:
    # Represents one game state: the board and the mark whose turn it is.

    def __init__(self, board=None, current_player=X_MARK):
        if board is None:
            board = [EMPTY] * BOARD_SIZE
        if len(board) != BOARD_SIZE:
            raise ValueError("A Tic-Tac-Toe board must contain exactly 9 positions.")
        if current_player not in PLAYER_MARKS:
            raise ValueError("current_player must be 'X' or 'O'.")
        if any(cell not in (EMPTY, X_MARK, O_MARK) for cell in board):
            raise ValueError("Board positions must contain 'X', 'O', or an empty space.")

        # Use a tuple so a state cannot be changed accidentally.
        self.board = tuple(board)
        self.current_player = current_player

    def legal_moves(self):
        # Return indices of all unoccupied positions.
        return [i for i, cell in enumerate(self.board) if cell == EMPTY]

    def apply_move(self, move):
        # Return a new state after a legal move, leaving this state unchanged.
        if move not in self.legal_moves():
            raise ValueError("Move must select an empty position on the board.")

        next_board = list(self.board)
        next_board[move] = self.current_player
        next_player = O_MARK if self.current_player == X_MARK else X_MARK
        return TicTacToeState(next_board, next_player)

    def winner(self):
        # Return the winning mark ('X' or 'O'), or None if there is no winner.
        for player in PLAYER_MARKS:
            if check_winner(self.board, player):
                return player
        return None

    def is_terminal(self):
        # Return True when the state is a win, a loss, or a draw.
        return self.winner() is not None or len(self.legal_moves()) == 0

In [ ]:
# Game class:

class TicTacToeGame:

    def __init__(self, human_mark=None):
        # Optionally set the human's mark to 'X' or 'O'.
        if human_mark is not None and human_mark not in PLAYER_MARKS:
            raise ValueError("human_mark must be 'X' or 'O'.")
        self.human_mark = human_mark
        self.ai_mark = self._opposite_mark(human_mark) if human_mark is not None else None

    @staticmethod
    def _opposite_mark(mark):
        # Return the other Tic-Tac-Toe mark.
        return O_MARK if mark == X_MARK else X_MARK

    def utility(self, state, depth):
        # Score a terminal state from the AI's perspective.
        winner = state.winner()
        if winner == self.ai_mark:
            return 10 - depth       # Prefer winning sooner.
        if winner == self.human_mark:
            return depth - 10       # Prefer delaying an unavoidable loss.
        return 0                    # A draw has a score of zero.

    def minimax(self, state, depth=0):
        # Return the best achievable score from this state using Minimax alone.
        if state.is_terminal():
            return self.utility(state, depth)

        if state.current_player == self.ai_mark:
            # The AI maximizes its own score, whether it plays X or O.
            best_score = float("-inf")
            for move in state.legal_moves():
                child = state.apply_move(move)
                score = self.minimax(child, depth + 1)
                best_score = max(best_score, score)
            return best_score

        # The human is assumed to choose moves that minimize the AI's score.
        best_score = float("inf")
        for move in state.legal_moves():
            child = state.apply_move(move)
            score = self.minimax(child, depth + 1)
            best_score = min(best_score, score)
        return best_score

    def choose_ai_move(self, state):
        legal_moves = state.legal_moves()
        if not legal_moves:
            return None

        best_move = legal_moves[0]
        best_score = float("-inf")
        for move in legal_moves:
            child = state.apply_move(move)
            score = self.minimax(child, depth=1)
            if score > best_score:
                best_score = score
                best_move = move

        return best_move

    def play(self):
        # Play a full console game; the human may choose X or O.
        if self.human_mark is None:
            while True:
                choice = input("Would you like to play as X or O? (X goes first): ").strip().upper()
                if choice in PLAYER_MARKS:
                    self.human_mark = choice
                    self.ai_mark = self._opposite_mark(choice)
                    break
                print("Please enter X or O.")

        state = TicTacToeState(current_player=X_MARK)  # X always begins.
        print(f"Welcome to TicTacThink! You are {self.human_mark}.")
        print("On your turn, choose a square by entering a number from 1 to 9.")
        if self.human_mark == O_MARK:
            print("X goes first, so the AI will make the opening move.")

        print_board(state.board)

        while not state.is_terminal():
            if state.current_player == self.human_mark:
                choice_text = input("Your move (1-9): ").strip()
                try:
                    move = int(choice_text) - 1
                except ValueError:
                    print("Invalid input. Enter a number from 1 to 9.")
                    continue

                if move not in state.legal_moves():
                    print("That square is invalid or already taken. Try again.")
                    continue

                state = state.apply_move(move)
                if state.is_terminal():
                    print_board(state.board)
            else:
                move = self.choose_ai_move(state)
                print(f"AI chooses position {move + 1}.")
                state = state.apply_move(move)
                print_board(state.board)  # Display the board after each AI move.

        winner = state.winner()
        if winner == self.ai_mark:
            print("The AI wins!")
        elif winner == self.human_mark:
            print("You win! Congratulations!")
        else:
            print("It's a draw!")

In [ ]:
# run game
if __name__ == "__main__":
    TicTacToeGame().play()

Would you like to play as X or O? (X goes first): X
Welcome to TicTacThink! You are X.
On your turn, choose a square by entering a number from 1 to 9.

 1 | 2 | 3 
---|---|---
 4 | 5 | 6 
---|---|---
 7 | 8 | 9 

Your move (1-9): 1
Score at position 2: -4
Score at position 3: -4
Score at position 4: -4
Score at position 5: 0
Score at position 6: -4
Score at position 7: -4
Score at position 8: -4
Score at position 9: -4
AI chooses position 5.

 X | 2 | 3 
---|---|---
 4 | O | 6 
---|---|---
 7 | 8 | 9 

Your move (1-9): 9
Score at position 2: 0
Score at position 3: -6
Score at position 4: 0
Score at position 6: 0
Score at position 7: -6
Score at position 8: 0
AI chooses position 2.

 X | O | 3 
---|---|---
 4 | O | 6 
---|---|---
 7 | 8 | X 

Your move (1-9): 3
Score at position 4: -8
Score at position 6: 7
Score at position 7: -8
Score at position 8: 9
AI chooses position 8.

 X | O | X 
---|---|---
 4 | O | 6 
---|---|---
 7 | O | X 

The AI wins!
